In [ ]:
from sympy import symbols
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# formualation for a_t(ask) and b_t(bid)

P_Buy_Noise = symbols('P_Buy_Noise_given_Omega_t_minus_1_v')
P_Buy = symbols('P_Buy_given_Omega_t_minus_1_v')
E_v_given_Omega = symbols('E_v_given_Omega_t_minus_1')
P_Buy_Informed = symbols('P_Buy_Informed_given_Omega_t_minus_1_v')
E_v_given_Omega_geq_at = symbols('E_v_given_Omega_t_minus_1_v_geq_at')

a_t = (P_Buy_Noise / P_Buy) * E_v_given_Omega + (P_Buy_Informed / P_Buy) * E_v_given_Omega_geq_at

P_Sell_Noise = symbols('P_Sell_Noise_given_Omega_t_minus_1_v')
P_Sell = symbols('P_Sell_given_Omega_t_minus_1_v')
E_v_given_Omega = symbols('E_v_given_Omega_t_minus_1')  # same as used in a_t
P_Sell_Informed = symbols('P_Sell_Informed_given_Omega_t_minus_1_v')
E_v_given_Omega_leq_at = symbols('E_v_given_Omega_t_minus_1_v_leq_at')


b_t = (P_Sell_Noise / P_Sell) * E_v_given_Omega + (P_Sell_Informed / P_Sell) * E_v_given_Omega_leq_at

In [ ]:
# optimal formulation of spread
S = a_t - b_t
S

E_v_given_Omega_t_minus_1*P_Buy_Noise_given_Omega_t_minus_1_v/P_Buy_given_Omega_t_minus_1_v - E_v_given_Omega_t_minus_1*P_Sell_Noise_given_Omega_t_minus_1_v/P_Sell_given_Omega_t_minus_1_v + E_v_given_Omega_t_minus_1_v_geq_at*P_Buy_Informed_given_Omega_t_minus_1_v/P_Buy_given_Omega_t_minus_1_v - E_v_given_Omega_t_minus_1_v_leq_at*P_Sell_Informed_given_Omega_t_minus_1_v/P_Sell_given_Omega_t_minus_1_v

Those variables should be adjusted accordingly by the data we have.

In [ ]:
# pi # the probability of informed trader
# beta_B # fixed probability of buying from noise traders
# beta_S # fixed probability of selling from noise traders
# P_Buy_Informed = pi * # Probability of a Buy by Informed trader given omiga_t_minus_1
# P_Sell_Informed = Pi * # Probability of a Sell by Informed trader given omiga_t_minus_1
# E_v_given_Omega_geq_at  # Expectation of v conditioned on v >= a_t, which is the previous bid/ask price (Known)
# E_v_given_Omega_leq_at  #  Expectation of v conditioned on v <= a_t, which is the current bid/ask price (Known)

# E_v_given_Omega # the corresponding bid/ask price at time t-1 , which is mu_t_minus_1

# P_Sell = (1 - pi) * beta_B + pi * P_Sell_Informed
# P_Buy = (1 - pi) * beta_S + pi * P_Buy_Informed
# P_Buy_Noise  = (1-pi) * beta_B  # Probability of a Buy by Noise trader :estimate from the historical data
# P_Sell_Noise = (1-pi) * beta_S   # Probability of a Sell by Noise trader: estimate from the histrocial data



In [ ]:
# example
values = {
    P_Buy_Noise: 0.5,  # Probability of a Buy by Noise trader
    P_Buy: 0.7,        # Total probability of observing Buy
    E_v_given_Omega: 100,  # Prior expectation of v (same for both a_t and b_t)
    P_Buy_Informed: 0.2,  # Probability of a Buy by Informed trader
    E_v_given_Omega_geq_at: 105,  # Expectation of v conditioned on v >= a_t, which is the previous bid/ask price (Known)

    # Values for b_t reused here
    P_Sell_Noise: 0.4,  # Probability of a Sell by Noise trader
    P_Sell: 0.6,        # Total probability of observing Sell
    P_Sell_Informed: 0.2,  # Probability of a Sell by Informed trader
    E_v_given_Omega_leq_at: 95  # Expectation of v conditioned on v <= a_t, which is the current bid/ask price (Known)
}

# Substitute and calculate both a_t and b_t
a_t_value = a_t.subs(values).evalf()
b_t_value = b_t.subs(values).evalf()

# Calculate the spread S = a_t - b_t
S = a_t_value - b_t_value
a_t_value, b_t_value, S

(101.428571428571, 98.3333333333333, 3.09523809523809)

# Assuming the real value is a Binary Tree

In [ ]:
# Assuming the real value is a Binary tree

# Define the symbols for theta, pi, beta_B, beta_S, and values v_H, v_L
theta = symbols('theta')  # Probability of V_high
pi = symbols('pi')  # Probability of informed trader
beta_B = symbols('beta_B')  # Probability of noise trader's buy order
beta_S = symbols('beta_S')  # Probability of noise trader's sell order
v_H = symbols('v_H')  # High value of v (value of asset)
v_L = symbols('v_L')  # Low value of v (value of asset)

# Define the formulas for (a - mu) and (mu - b)
a_minus_mu = (theta * (1 - theta) * pi) / ((1 - pi) * beta_B + pi * theta) * (v_H - v_L)
mu_minus_b = (theta * (1 - theta) * pi) / ((1 - pi) * beta_S + pi * (1 - theta)) * (v_H - v_L)

# Define values for the symbols for evaluation
example_values = {
    theta: 0.4,  # Proportion of informed traders (40% informed)
    pi: 0.3,  # Probability of informed trader in market
    beta_B: 0.6,  # Probability of noise trader issuing buy orders
    beta_S: 0.5,  # Probability of noise trader issuing sell orders
    v_H: 120,  # High true value
    v_L: 100   # Low true value
}

# Substitute values in both equations
a_minus_mu_result = a_minus_mu.subs(example_values).evalf()
mu_minus_b_result = mu_minus_b.subs(example_values).evalf()

a_minus_mu_result, mu_minus_b_result, a_minus_mu_result - mu_minus_b_result


(2.66666666666667, 2.71698113207547, -0.0503144654088068)

In [ ]:
import pandas as pd


df = pd.read_excel('NEW__ RFQ US Rates - Bonds Only (July - Sept 24) - Blockhouse.xlsx')
df.head()


,EventDate,Id,Client,RFQStatus,InstrumentSubGroup,Mkt Traded Vol,Market Traded Vol (USD),Our Traded Vol,Our Traded Vol (USD),TiedWonVol,...,SettlementDays,SettlementDate,SettlementStops,RFQ CreateTime,RFQ ClosedTime,AnAutoReplyRFQs,First Response Time (Sec),TradeSizeBucket,Sector,MaturityBucket
0,2024-09-30,66FA7D45432800D20002,Client8,Done,OFFTHERUN,100000000,100000000,100000000,100000000,100000000,...,1,2024-10-01,T+1,2024-09-30 06:28:00,2024-09-30 06:28:00,1,0.03,50+M,5-7 YR,5y-7y
1,2024-09-30,TRSY_20240930_15961,Client9,Done,"OFFTHERUN,DOUBLEOLDS",2100000,2100000,2100000,2100000,2100000,...,1,2024-10-01,T+1,2024-09-30 16:00:00,2024-09-30 16:00:00,1,0.95,1-5M,7-10 YR,7y-10y
2,2024-09-30,TRSY_20240930_314,Client9,Done,ONTHERUN,53000000,53000000,53000000,53000000,53000000,...,1,2024-10-01,T+1,2024-09-30 15:46:00,2024-09-30 15:46:00,1,0.07,50+M,7-10 YR,7y-10y
3,2024-09-30,66FAEA57432800020001,Client41,Done,OFFTHERUN,500000,500000,500000,500000,500000,...,1,2024-10-01,T+1,2024-09-30 14:14:00,2024-09-30 14:14:00,1,0.03,500K-1M,0-1 YR,<18mos
4,2024-09-30,66FAC35D455C00210006,Client139,Done,OFFTHERUN,1058000,1058000,1058000,1058000,1058000,...,1,2024-10-01,T+1,2024-09-30 11:27:00,2024-09-30 11:27:00,1,0.03,1-5M,5-7 YR,3y-5y


In [ ]:
maturity_buckets = df['MaturityBucket'].unique()

# List of unique tiers
tiers = df['Tier'].unique()

# Dictionary to hold the datasets
datasets = {}

# Loop through each maturity bucket and tier to create a separate dataset
for maturity in maturity_buckets:
    for tier in tiers:
        # Create a key for the dictionary in the format "Maturity_Tier"
        key = f"{maturity}_{tier}"
        # Filter the dataframe for the specific maturity and tier
        datasets[key] = df[(df['MaturityBucket'] == maturity) & (df['Tier'] == tier)]



In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Log-likelihood function for MLE
def log_likelihood(params, prices, labels):
    V_high, V_low, theta, sigma_high, sigma_low = params

    # Probability density functions
    prob_high = (1 / (np.sqrt(2 * np.pi) * sigma_high)) * np.exp(-0.5 * ((prices - V_high) / sigma_high) ** 2)
    prob_low = (1 / (np.sqrt(2 * np.pi) * sigma_low)) * np.exp(-0.5 * ((prices - V_low) / sigma_low) ** 2)

    # Mixture model
    mixture_prob = theta * prob_high + (1 - theta) * prob_low

    # Negative log-likelihood
    ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
    return -ll

# Function to run MLE on a dataset
def run_mle(data):
    # Mark invalid Best Bid and Best Ask as abstain
    data.loc[(data['Best Bid Price'] == 0) & (data['Best Ask Price'] == 0), 'Buy/Sell'] = 'abstrain'

    # Filter valid rows
    filtered_data = data[(data['Best Bid Price'] != 0) & (data['Best Ask Price'] != 0)]

    # Prepare data for MLE
    bid_prices = filtered_data[filtered_data['Buy/Sell'] == 'Sell']['Best Bid Price'].values
    ask_prices = filtered_data[filtered_data['Buy/Sell'] == 'Buy']['Best Ask Price'].values
    buy_sell_prices = filtered_data['Deal Value'].values
    labels = np.where(filtered_data['Buy/Sell'] == 'Buy', 1, 0)

    # Initialize parameters
    V_high_init = np.mean(ask_prices)
    V_low_init = np.mean(bid_prices)
    initial_guess = [V_high_init, V_low_init, 0.5, 1.0, 1.0]
    bounds = [(None, None), (None, None), (0, 1), (1e-3, None), (1e-3, None)]

    # Optimize MLE
    result = minimize(log_likelihood, initial_guess, args=(buy_sell_prices, labels), bounds=bounds)

    # Calculate value counts and percentages
    buy_sell_counts = data['Buy/Sell'].value_counts()
    total_count = buy_sell_counts.sum()
    percentages = (buy_sell_counts / total_count) * 100

    # Return MLE results and percentages
    return {
        'V_high': result.x[0],
        'V_low': result.x[1],
        'theta': result.x[2],
        #'sigma_high': result.x[3],
        #'sigma_low': result.x[4],
        #'success': result.success,
        'Buy %': percentages.get('Buy', 0)/100,
        'Sell %': percentages.get('Sell', 0)/100,
        'Abstrain %': percentages.get('abstrain', 0)/100
    }

# Iterate over datasets dictionary and apply MLE
mle_results = {}
for key, subset in datasets.items():
    mle_results[key] = run_mle(subset)




<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value encountered in log
  ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value encountered in log
  ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value encountered in log
  ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value encountered in log
  ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value encountered in log
  ll = np.sum(labels * np.log(mixture_prob + 1e-9) + (1 - labels) * np.log(1 - mixture_prob + 1e-9))
<ipython-input-10-fa20b0d3f9e2>:17: RuntimeWarning: invalid value enco

In [ ]:
mle_results_df = pd.DataFrame(mle_results).T

initial_guesses = {}

for key, data in datasets.items():

    bid_prices = data[(data['Buy/Sell'] == 'Sell') & (data['Best Bid Price'] != 0)]['Best Bid Price'].values
    ask_prices = data[(data['Buy/Sell'] == 'Buy') & (data['Best Ask Price'] != 0)]['Best Ask Price'].values


    V_high_init = np.mean(ask_prices) if len(ask_prices) > 0 else 0
    V_low_init = np.mean(bid_prices) if len(bid_prices) > 0 else 0


    initial_guesses[key] = {
        'V_high': V_high_init,
        'V_low': V_low_init,
        'theta': 0.5,
        'sigma_high': 1.0,
        'sigma_low': 1.0
    }



# Replace NaN values in the MLE results DataFrame
for param, init_value in initial_guesses.items():
    if param in mle_results_df.columns:
        mle_results_df[param].fillna(init_value, inplace=True)

# Check the updated DataFrame
mle_results_df.sort_index(inplace=True)



# Reading Informed Trader percentages from Jeffery's Report


In [ ]:
import pandas as pd

# # Load the uploaded Excel file to inspect its contents
# file_path = 'Jefferies Report.xlsx'
# excel_data = pd.ExcelFile(file_path)

# # Display the sheet names to understand the structure of the file
# excel_data.sheet_names

# trading_overview = excel_data.parse('Trading Overview')
# sample_insight = excel_data.parse('Sample Insight')
# on_the_run = excel_data.parse('On-the-run')
# off_the_run = excel_data.parse('Off-the-run')

In [ ]:
mle_results_df

,V_high,V_low,theta,Buy %,Sell %,Abstrain %
10-30y_MANUAL,90.724884,91.183057,0.000000,0.464770,0.437669,0.097561
10-30y_TIER1,94.606754,94.224874,0.000000,0.450256,0.500513,0.049231
10-30y_TIER2,96.148370,96.090397,0.000000,0.428435,0.552290,0.019275
10-30y_TIER3,102.254974,93.952895,1.000000,0.274194,0.516129,0.209677
10-30y_TIER9,96.492424,102.928936,0.000000,0.243655,0.553299,0.203046
10-30y_TierGOLD,104.179815,99.698057,0.866299,0.500000,0.484375,0.015625
10-30y_TierINTERNAL,103.643670,100.183456,0.810068,0.454068,0.530184,0.015748
10-30y_UNKNOWN,54.187500,89.220703,0.500000,0.200000,0.800000,0.000000
18m-2y_MANUAL,96.622796,101.327306,0.635293,0.259615,0.600962,0.139423
18m-2y_TIER1,100.491572,95.008715,0.716215,0.444730,0.544987,0.010283


In [ ]:
index = mle_results_df.index

# The corresponding informed trader ratios from on the run sheet
informed_trader_values = [
    0.41, 0.66, 0.35, 0.67, 0.67, 0.50, 0.33, np.nan,
    0.56, 0.7, 0.31, 1, 1, 1, 0.50, np.nan,
    0.83, 0.7, 0.41, 1, 0.5, 1, 1, np.nan,
    0.48, 0.71, 0.33, 1, 0.71, 0.5, 0.50, np.nan,
    0.57, 0.7, 0.38, 0.67, 0.80, 0.5, 0.50, np.nan,
    0.39, 0.61, 0.27, 0.67, 0.75, 0.5, 0.33, np.nan,
    np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
]

# Create the final DataFrame
informed_trader_df = pd.DataFrame({"Informed Trader %": informed_trader_values}, index=index)
mle_results_df_on_run = pd.concat([mle_results_df, informed_trader_df], axis=1)



In [ ]:
for row in mle_results_df_on_run.iterrows():

    example_values = {
      theta: row[1]['theta'],  # Proportion of informed traders
      pi: row[1]['Informed Trader %'],  # Probability of informed trader in market
      beta_B: row[1]['Buy %'],  # Probability of noise trader issuing buy orders
      beta_S: row[1]['Sell %'],  # Probability of noise trader issuing sell orders
      v_H: row[1]['V_high'],  # High true value
      v_L: row[1]['V_low']   # Low true value
    }

    #print(example_values)
    # Substitute values in both equations
    a_minus_mu_result = a_minus_mu.subs(example_values).evalf()
    mu_minus_b_result = mu_minus_b.subs(example_values).evalf()
    print(row[0] , (a_minus_mu_result + mu_minus_b_result))

10-30y_MANUAL 0
10-30y_TIER1 0
10-30y_TIER2 0
10-30y_TIER3 0
10-30y_TIER9 0
10-30y_TierGOLD 1.21979615922686
10-30y_TierINTERNAL 0.727786340153440
10-30y_UNKNOWN nan
18m-2y_MANUAL -2.60121341609809
18m-2y_TIER1 3.38296070242227
18m-2y_TIER2 -1.80827418530176
18m-2y_TIER3 -0.0671844000000021
18m-2y_TIER9 nan
18m-2y_TierGOLD -0.0235322683662247
18m-2y_TierINTERNAL -0.516485456217854
18m-2y_UNKNOWN nan
2y-3y_MANUAL -5.14171719030999
2y-3y_TIER1 4.27767066303701
2y-3y_TIER2 -2.31452074555099
2y-3y_TIER3 nan
2y-3y_TIER9 0
2y-3y_TierGOLD 2.00249110066000
2y-3y_TierINTERNAL -0.754613288704292
2y-3y_UNKNOWN nan
3y-5y_MANUAL -2.23120113690906
3y-5y_TIER1 0
3y-5y_TIER2 -1.71471522065485
3y-5y_TIER3 0.0414865380434719
3y-5y_TIER9 -0.429360102610628
3y-5y_TierGOLD 0.209734586641247
3y-5y_TierINTERNAL 0.540785690209695
3y-5y_UNKNOWN nan
5y-7y_MANUAL 4.99261195757535
5y-7y_TIER1 -4.65018655196862
5y-7y_TIER2 0
5y-7y_TIER3 -0.904176262538824
5y-7y_TIER9 2.59273502704400
5y-7y_TierGOLD 0.7430151834859

In [ ]:
informed_trader_values_off_run = [
    0.57, 0.65, 0.46, 0, 0.5, 0.0, 0.00, np.nan,
    0.68, 0.64, 0.53, 0, 0.67, 0, 0.00, np.nan,
    0.5, 0.57, 0.51, 0, 1, 0, 0.00, np.nan,
    0.52, 0.58, 0.46, 0, 0.6, 0, 0.00, np.nan,
    0.63, 0.64, 0.57, 0, 1, 0, 0.00, np.nan,
    0.68, 0.69, 0.51, 0, 0.5, 0.00, 0.00, np.nan,
    0.39, 0.45, 0.34, 0, 0.6, 0, 0, np.nan
]
informed_trader_df_off_run = pd.DataFrame({"Informed Trader %": informed_trader_values_off_run}, index=index)
mle_results_df_off_run = pd.concat([mle_results_df, informed_trader_df_off_run], axis=1)

In [ ]:
for row in mle_results_df_off_run.iterrows():

    example_values = {
      theta: row[1]['theta'],  # Proportion of informed traders
      pi: row[1]['Informed Trader %'],  # Probability of informed trader in market
      beta_B: row[1]['Buy %'],  # Probability of noise trader issuing buy orders
      beta_S: row[1]['Sell %'],  # Probability of noise trader issuing sell orders
      v_H: row[1]['V_high'],  # High true value
      v_L: row[1]['V_low']   # Low true value
    }

    #print(example_values)
    # Substitute values in both equations
    a_minus_mu_result = a_minus_mu.subs(example_values).evalf()
    mu_minus_b_result = mu_minus_b.subs(example_values).evalf()
    print(row[0] , (a_minus_mu_result + mu_minus_b_result))

In [ ]:
mle_results_df_off_run